<a href="https://colab.research.google.com/github/meliikekacar/n8n_no_code_automation/blob/main/hw4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
import json

def parse_ticket_line(line: str) -> dict:

    line = re.sub(r"\t+", " ", line.strip())

    segments = re.split(r"\s*[|;,]\s*", line)

    data = {}

    FIELD_MAP = {
        "name": "name",
        "fullname": "name",
        "email": "email",
        "mail": "email",
        "phone": "phone",
        "tel": "phone",
        "severity": "severity",
        "sev": "severity",
        "note": "note",
        "memo": "note"
    }

    for seg in segments:

        if re.fullmatch(r"TCK-\d{6}", seg):
            data["ticket_id"] = seg
            continue


        if re.fullmatch(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}", seg):
            data["timestamp"] = seg
            continue


        m = re.match(r"(\w+)\s*[:=]\s*(.+)", seg)
        if m:
            key, value = m.group(1).lower(), m.group(2).strip()
            if key in FIELD_MAP:
                data[FIELD_MAP[key]] = value


    required_fields = ["ticket_id", "timestamp", "email", "phone", "severity", "note"]
    for field in required_fields:
        if field not in data:
            raise ValueError(f"Missing field: {field}")


    if not re.fullmatch(r"TCK-\d{6}", data["ticket_id"]):
        raise ValueError("Invalid ticket_id")


    if not re.fullmatch(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}", data["timestamp"]):
        raise ValueError("Invalid timestamp")


    if not re.fullmatch(r"[^@]+@[^@]+\.[^@]+", data["email"]):
        raise ValueError("Invalid email")


    phone = re.sub(r"[^\d+]", "", data["phone"])
    if not phone.startswith("+"):
        phone = "+" + phone
    data["phone"] = phone


    severity = data["severity"].upper()
    if severity not in {"P1", "P2", "P3", "P4"}:
        raise ValueError("Invalid severity")
    data["severity"] = severity


    urls = re.findall(r"https?://\S+", data["note"])
    data["urls"] = urls

    return data


# ================= MAIN PIPELINE =================

input_file = "/content/tickets_raw.txt"

valid_tickets = []
invalid_lines = []

severity_count = {"P1": 0, "P2": 0, "P3": 0, "P4": 0}
domain_count = {}
total_urls = 0
total_lines = 0

try:
    with open(input_file, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            total_lines += 1
            try:
                ticket = parse_ticket_line(line)
                valid_tickets.append(ticket)

                severity_count[ticket["severity"]] += 1
                domain = ticket["email"].split("@")[1]
                domain_count[domain] = domain_count.get(domain, 0) + 1
                total_urls += len(ticket["urls"])

            except ValueError as e:
                invalid_lines.append(
                    f"LINE {line_no}: {e} | {line.strip()}"
                )

except (FileNotFoundError, PermissionError) as e:
    print("File error:", e)

finally:
    print("ETL job finished.")


# ================= OUTPUT FILES =================

with open("/content/tickets_valid.json", "w", encoding="utf-8") as jf:
    json.dump(valid_tickets, jf, indent=4, ensure_ascii=False)

with open("/content/tickets_invalid.txt", "w", encoding="utf-8") as af:
    af.write("\n".join(invalid_lines))

top_domains = sorted(
    domain_count.items(),
    key=lambda x: x[1],
    reverse=True
)[:3]

with open("/content/kpi_report.txt", "w", encoding="utf-8") as kf:
    kf.write(f"Total lines processed: {total_lines}\n")
    kf.write(f"Valid tickets: {len(valid_tickets)}\n")
    kf.write(f"Invalid tickets: {len(invalid_lines)}\n\n")

    kf.write("Count by severity:\n")
    for k, v in severity_count.items():
        kf.write(f"{k}: {v}\n")

    kf.write("\nTop 3 email domains:\n")
    for d, c in top_domains:
        kf.write(f"{d}: {c}\n")

    kf.write(f"\nTotal URLs found: {total_urls}\n")



ETL job finished.


In [ ]:
%%writefile README.md
AIN-1001 – Homework 4
Student ID: XXXXXXXX

Description
This project implements a mini ETL pipeline that processes a raw helpdesk
ticket export file using regular expressions, file handling, and exception
management.

How to Run
The program is executed using Python 3:

python hw4.py tickets_raw.txt

Assumptions
- Field order is not fixed.
- Different field names (name/fullname, email/mail, etc.) are accepted.
- Lines missing required fields are rejected.
- Only severity levels P1–P4 are valid.

Regex Usage
- re.sub(): normalize tabs and phone numbers
- re.split(): split inconsistent delimiters
- re.fullmatch(): validate ticket_id, timestamp, and email
- re.findall(): extract URLs from note

Outputs
- tickets_valid.json
- tickets_invalid.txt
- kpi_report.txt


Writing README.md
